# ISU Campus Assistant — learn a reward head from your preferences (RLHF)

**ReggieBot's search → the same 8 excerpts → two Gemma 4 answers (A short, B detailed) → your choice →
a small learned reward score.**

This notebook adds the first stage of RLHF to ReggieBot, following the IT244 reward-head notebook:

| | |
|---|---|
| **Evidence** | Only the sources ReggieBot has already collected: the ISU course catalog and the billing, housing and dining pages saved in `isu_catalog_cache/`. The search is the chat app's own (BM25 + Chroma). There is no web search and no Wikipedia |
| **Two answers** | Gemma 4 (`gemma4:12b` in Ollama) writes both from the **same** evidence: **A is short and simple, B explains more** |
| **Your choice** | Prefer A, prefer B, or skip. Saved to `experiments/rlhf/preferences.jsonl` |
| **What learns** | A small reward head: one linear layer with **896 weights**, trained with the pairwise Bradley–Terry loss |
| **What stays frozen** | Gemma 4 (the answer writer) and Qwen2.5-0.5B (the small encoder the reward head reads) |

The reward head reads **question + evidence + answer** through a frozen Qwen2.5-0.5B. It would be more
direct to read Gemma 4 itself, but this Ollama version cannot return Gemma 4's internal features, so a small,
separate encoder does that job. That is the usual RLHF setup: the reward model is its own network.

Reward modeling is the first stage of RLHF; by itself it does not retrain Gemma 4. §6 uses the trained head
the simplest way that changes what users see: it writes both answers and **shows the one the head predicts you
prefer** (best-of-2). Retraining Gemma 4 itself, with LoRA and a policy-gradient method such as RLOO, is the
next stage and is not done here.

---
## 1 · Setup

Start Ollama first. This loads the saved data (no crawling) and checks that Gemma 4 is ready.

In [1]:
import hashlib
import io
import json
import os
import random
import sys
import time
from collections import Counter
from contextlib import redirect_stdout
from datetime import datetime, timezone
from pathlib import Path

# Work from the project folder - the nearest folder holding reggiebot/ - like the other notebooks.
PROJECT = next((folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "reggiebot").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("No reggiebot/ folder here or above - keep this notebook inside the project folder.")
sys.path.insert(0, str(PROJECT))

from reggiebot.answer import SYSTEM_PROMPT
from reggiebot.config import CACHE_DIR, OLLAMA_MODEL, TOP_K
from reggiebot.core import CatalogApp

# --- settings -------------------------------------------------------------------------------------------
ENCODER_ID      = "Qwen/Qwen2.5-0.5B-Instruct"   # the small frozen model the reward head reads (~1 GB, CPU)
EVIDENCE_TOKENS = 768     # evidence the reward encoder reads per answer (the IT244 notebook used 640)
HELDOUT_SHARE   = 0.25    # about 1 question in 4 is kept out of training, decided by the question itself
DATA  = CACHE_DIR / "experiments" / "rlhf"
PREFS = DATA / "preferences.jsonl"
DATA.mkdir(parents=True, exist_ok=True)

APP = CatalogApp()
APP.load(progress=lambda msg: print("  ", msg))
print("\nanswers come from:", APP.answerer.model if APP.answerer.available else f"NOT READY - {APP.answerer.error}")
print("preferences are saved to:", PREFS)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   loaded 6797 saved documents from C:\Users\Jake\Downloads\ISU-RAG-VSCode\isu_catalog_cache - no crawling
   building keyword index
   vector DB already current - 6797 vectors
   index ready - 6797 courses
[ollama] using gemma4:12b at http://127.0.0.1:11434 (context 8,192 tokens)

answers come from: gemma4:12b
preferences are saved to: C:\Users\Jake\Downloads\ISU-RAG-VSCode\isu_catalog_cache\experiments\rlhf\preferences.jsonl


---
## 2 · Two answers from the same evidence: A short, B detailed

`retrieve()` runs ReggieBot's own search over the saved ISU data and builds ReggieBot's own prompt: the rules
in `SYSTEM_PROMPT` ("use ONLY the catalog excerpts in CONTEXT", cite sources, never invent figures) plus the 8
excerpts. Nothing outside `isu_catalog_cache/` can reach the model.

`write()` asks Gemma 4 for each answer with one extra line of instructions that sets the style. Everything else
is identical for A and B, so your choice is about how the answer is written, not about what it was given.

In [2]:
STYLES = {
    "short": ("A · short and simple",
              "STYLE FOR THIS ANSWER - this overrides the length rule above: answer in one or two plain "
              "sentences, under 50 words. Give only the direct answer, with no lists and no extra background. "
              "Still name the course code or the page you used."),
    "detailed": ("B · explains more",
                 "STYLE FOR THIS ANSWER - this overrides the length rule above: explain more fully, in about 120 "
                 "to 200 words. Start with the direct answer, then add the relevant details from CONTEXT - "
                 "requirements, credit hours, dates, options or steps - as a short paragraph or a few bullet "
                 "points. Add nothing that is not in CONTEXT."),
}


def retrieve(question):
    """The top 8 chunks from ReggieBot's saved sources, and the prompt built from them. No web, no Wikipedia."""
    question = " ".join(question.split())
    if not question:
        raise ValueError("Enter a question first.")
    hits = APP.index.search(question, k=TOP_K)
    if not hits:
        raise ValueError("Nothing in the saved ISU data matched that. Try a course code or a campus topic.")
    return {
        "question": question,
        "prompt": APP.answerer.build_prompt(question, hits),
        "evidence": "\n\n".join(f"[{i}] {hit['doc']['text']}" for i, hit in enumerate(hits, 1)),
        "sources": [{"id": hit["doc"]["id"], "title": hit["doc"]["metadata"].get("title", ""),
                     "url": hit["doc"]["metadata"]["url"]} for hit in hits],
        "retrieved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "model": APP.answerer.model,
    }


def write(item, style, attempts=2):
    """One answer from Gemma 4, in the given style, from exactly the evidence in item. Retries once,
    because Ollama occasionally restarts its model runner (for example when the GPU is short of memory)."""
    system = SYSTEM_PROMPT + "\n\n" + STYLES[style][1]
    for attempt in range(1, attempts + 1):
        try:
            if not APP.answerer.available and not APP.answerer.connect():
                raise RuntimeError(APP.answerer.error)
            return "".join(APP.answerer.backend.generate(system, item["prompt"])).strip()
        except Exception as exc:
            if attempt == attempts:
                raise RuntimeError(f"Gemma 4 could not write the {style} answer ({str(exc)[:160]}). Check that "
                                   "Ollama is running, close other programs that use the GPU, and try again.") from exc
            time.sleep(3)


def sources_markdown(item):
    """Each source page once, in the order the search ranked them (several excerpts can share a page)."""
    pages = {}
    for s in item["sources"]:
        pages.setdefault(s["url"], s["title"] or s["id"])
    return "\n".join(f"{i}. [{title}]({url})" for i, (url, title) in enumerate(pages.items(), 1))


sample = retrieve("What are the prerequisites for IT 214?")
print("search found:", ", ".join(s["id"] for s in sample["sources"][:4]), "...")

search found: IT214, THE366A01, MSC240, HIS469 ...


---
## 3 · Collect your preferences

Run the cell, then use the app. For each question:

1. Pick a question, or press **Next suggested question**.
2. Press **Generate A / B**. A is always the short answer, B always the longer one.
3. Choose the answer you would rather get from ReggieBot. Judge it by whether it is **supported by the
   sources listed under it**, honest about what the sources do not say, and **useful**: sometimes one line is
   all you need, sometimes the details matter. Choose **Neither** when neither deserves it; that pair is skipped.

**Held-out questions are chosen for you.** Each question is assigned to `train` or `heldout` from its own
wording (about 1 in 4 is held out), before you see any answers, so the same question always lands in the same
split. Rewordings of a question can land in different splits, so rate each idea once.

Two training pairs and one held-out pair are the minimum; about **12 questions in total** gives a classroom
experiment. That is still far too few to prove a quality gain. Questions, sources and answers are saved; no
names or IDs.

In [3]:
os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
import gradio as gr

FALLBACK_QUESTIONS = ["What are the prerequisites for IT 214?", "How do I sign up for a payment plan?",
                      "Do first-year students have to live on campus?", "What are Flex Dollars?",
                      "Which courses teach database design?", "When will I get my refund?"]


def load_records():
    if not PREFS.exists():
        return []
    return [json.loads(line) for line in PREFS.read_text(encoding="utf-8").splitlines() if line.strip()]


def question_key(question):
    return " ".join(question.casefold().split()).rstrip("?.!")


def split_for(question):
    """train or heldout, fixed by the question's own wording."""
    bucket = int(hashlib.sha1(question_key(question).encode("utf-8")).hexdigest(), 16) % 1000
    return "heldout" if bucket < HELDOUT_SHARE * 1000 else "train"


def suggested_questions():
    """The 50 Arm 1 questions from the Ollama notebook, if it is in the project folder."""
    try:
        cells = json.loads((PROJECT / "ISU-CampusAssistant-RAG-Ollama.ipynb").read_text(encoding="utf-8"))["cells"]
        source = next("".join(c["source"]) for c in cells if "".join(c["source"]).startswith("# The Arm 1 question set."))
        ns = {"Counter": Counter}
        with redirect_stdout(io.StringIO()):
            exec(source, ns)
        return [q["question"] for q in ns["ARM1_QUESTIONS"]]
    except Exception:
        return FALLBACK_QUESTIONS


records = load_records()
SUGGESTED = suggested_questions()


def counts():
    by_split = Counter(r["split"] for r in records)
    by_style = Counter(r["chosen_style"] for r in records)
    return (f"{len(records)} preferences saved: {by_split['train']} training, {by_split['heldout']} held-out. "
            f"You chose A (short) {by_style['short']} times and B (explains more) {by_style['detailed']} times.")


def next_question(current):
    rated = {question_key(r["question"]) for r in records}
    for q in SUGGESTED:
        if question_key(q) not in rated and question_key(q) != question_key(current or ""):
            return q
    return random.choice(SUGGESTED)


def new_pair(question):
    try:
        return _new_pair(question)
    except gr.Error:
        raise
    except Exception as exc:                       # show the reason in the app, not a traceback
        raise gr.Error(str(exc)[:300])


def _new_pair(question):
    item = retrieve(question)
    item["split"] = split_for(item["question"])
    item["short"], item["detailed"] = write(item, "short"), write(item, "detailed")
    if not item["short"] or not item["detailed"] or item["short"] == item["detailed"]:
        raise gr.Error("The two answers were empty or identical. Press Generate again.")
    words = lambda text: len(text.split())
    a = f"### {STYLES['short'][0]}  ({words(item['short'])} words)\n\n{item['short']}"
    b = f"### {STYLES['detailed'][0]}  ({words(item['detailed'])} words)\n\n{item['detailed']}"
    seen = sum(1 for r in records if question_key(r["question"]) == question_key(item["question"]))
    note = f"This question is in the **{item['split']}** split." + (f" You have rated it {seen} time(s) before." if seen else "")
    return a, b, "**Sources both answers were given**\n\n" + sources_markdown(item), item, note


def prefer(choice, pending):
    if pending is None:
        return None, "Generate a pair first; each pair can be rated once."
    if choice is not None:
        other = "detailed" if choice == "short" else "short"
        row = {k: v for k, v in pending.items() if k not in ("short", "detailed")}
        row.update(short=pending["short"], detailed=pending["detailed"], chosen_style=choice,
                   chosen=pending[choice], rejected=pending[other], encoder=ENCODER_ID)
        with PREFS.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        records.append(row)
    return None, ("Saved. " if choice else "Skipped. ") + counts() + " Press Next suggested question."


with gr.Blocks(title="ReggieBot preferences") as rlhf_app:
    gr.Markdown("# Teach ReggieBot your preference: A or B")
    gr.Markdown("Both answers come from the **same** ISU sources, listed underneath. A is short and simple; "
                "B explains more. Choose the one you would rather get.")
    with gr.Row():
        question = gr.Textbox(value=next_question(""), label="Question", scale=5)
        suggest = gr.Button("Next suggested question", scale=1)
    generate = gr.Button("Generate A / B", variant="primary")
    with gr.Row():
        answer_a, answer_b = gr.Markdown(), gr.Markdown()
    sources = gr.Markdown()
    pending = gr.State(None)
    status = gr.Markdown(counts())
    with gr.Row():
        buttons = [gr.Button("Prefer A (short)"), gr.Button("Prefer B (explains more)"), gr.Button("Neither: skip")]
    suggest.click(next_question, [question], [question])
    generate.click(new_pair, [question], [answer_a, answer_b, sources, pending, status],
                   concurrency_id="gemma", concurrency_limit=1)
    for button, choice in zip(buttons, ("short", "detailed", None)):
        button.click(lambda p, c=choice: prefer(c, p), [pending], [pending, status], concurrency_id="gemma")

rlhf_app.queue(default_concurrency_limit=1).launch(inbrowser=False, quiet=True)
print(counts())

12 preferences saved: 9 training, 3 held-out. You chose A (short) 10 times and B (explains more) 2 times.


---
## 4 · The frozen encoder and the small reward head

The frozen Qwen2.5-0.5B reads **question + evidence + candidate answer** as one chat and turns it into one
vector: the hidden state of its last token, normalized to unit length. A linear head with no bias maps that
vector to one score. **Only the head's 896 weights learn.**

The evidence is included so the head can learn whether an answer fits what the sources say, not only how it
reads. It is cut to `EVIDENCE_TOKENS` (768) for speed on the CPU; the answers themselves were written from all
8 excerpts. Features are cached in `experiments/rlhf/features.pt`, so retraining is quick. The first run
downloads the encoder (~1 GB).

In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

enc_tokenizer = AutoTokenizer.from_pretrained(ENCODER_ID)
encoder = AutoModelForCausalLM.from_pretrained(ENCODER_ID, dtype=torch.float32).eval().requires_grad_(False)
FEATURE_CACHE = DATA / "features.pt"
feature_cache = torch.load(FEATURE_CACHE, weights_only=True) if FEATURE_CACHE.exists() else {}


@torch.no_grad()
def reward_features(question, evidence, answer):
    """The frozen encoder's view of one question + evidence + answer, as a unit-length vector."""
    key = hashlib.sha1("\x1f".join([ENCODER_ID, str(EVIDENCE_TOKENS), question, evidence, answer]).encode()).hexdigest()
    if key not in feature_cache:
        evidence_ids = enc_tokenizer.encode(evidence, add_special_tokens=False)[:EVIDENCE_TOKENS]
        messages = [{"role": "system", "content": "Answer using only the retrieved evidence."},
                    {"role": "user", "content": f"Question: {question}\n\nRetrieved evidence:\n"
                                                + enc_tokenizer.decode(evidence_ids, skip_special_tokens=True)},
                    {"role": "assistant", "content": answer}]
        ids = enc_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False,
                                                return_tensors="pt", return_dict=True)
        hidden = encoder(**ids, output_hidden_states=True, use_cache=False).hidden_states[-1][0, -1]
        feature_cache[key] = F.normalize(hidden.float(), dim=0)
    return feature_cache[key]


print(f"encoder ready: {ENCODER_ID}, hidden size {encoder.config.hidden_size} "
      f"-> a reward head of {encoder.config.hidden_size} weights")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

encoder ready: Qwen/Qwen2.5-0.5B-Instruct, hidden size 896 -> a reward head of 896 weights


---
## 5 · Train the head from your chosen/rejected pairs

**Bradley–Terry loss:** `-log(sigmoid(score_chosen - score_rejected))`, averaged over the training pairs.
Held-out pairs never enter the loss. The head starts from the same seed every time, so this cell can simply be
run again after rating more pairs. The epoch count is fixed; do not tune it on the held-out pairs.

Because A is always the short answer, there is an easy rule to beat: "always prefer whichever style you chose
more often". The report shows that baseline next to the head. **A head that does not beat it has only learned
your favourite length.** One that does has learned when you want the short answer and when you want detail.

In [5]:
def pair_features(rows):
    good = torch.stack([reward_features(r["question"], r["evidence"], r["chosen"]) for r in rows])
    bad = torch.stack([reward_features(r["question"], r["evidence"], r["rejected"]) for r in rows])
    return good, bad


def preference_accuracy(head, good, bad):
    with torch.no_grad():
        return (head(good).flatten() > head(bad).flatten()).float().mean().item()


records = load_records()
train_rows = [r for r in records if r["split"] == "train"]
test_rows = [r for r in records if r["split"] == "heldout"]
head, report = None, None

if len(train_rows) < 2 or len(test_rows) < 1:
    print(f"Not enough preferences yet: {len(train_rows)} training and {len(test_rows)} held-out. Rate more in §3 "
          "(at least 2 and 1; about 9 and 3 is better), then run this cell again.")
elif {question_key(r["question"]) for r in train_rows} & {question_key(r["question"]) for r in test_rows}:
    print("A question appears in both splits - check preferences.jsonl.")
else:
    good, bad = pair_features(train_rows)
    test_good, test_bad = pair_features(test_rows)
    torch.save(feature_cache, FEATURE_CACHE)

    torch.manual_seed(7)
    head = torch.nn.Linear(encoder.config.hidden_size, 1, bias=False)
    before = preference_accuracy(head, test_good, test_bad)
    optimizer = torch.optim.AdamW(head.parameters(), lr=0.01, weight_decay=0.01)
    for epoch in range(100):
        loss = -F.logsigmoid(head(good) - head(bad)).mean()    # Bradley-Terry pairwise preference loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    head.eval().requires_grad_(False)

    with torch.no_grad():
        scores = head(torch.cat([good, bad])).flatten()
        reward_mean, reward_std = scores.mean().item(), max(scores.std(unbiased=False).item(), 0.1)
    favourite = Counter(r["chosen_style"] for r in train_rows).most_common(1)[0][0]
    report = {
        "train_pairs": len(train_rows), "heldout_pairs": len(test_rows),
        "train_preference_accuracy": preference_accuracy(head, good, bad),
        "heldout_before_training": before,
        "heldout_after_training": preference_accuracy(head, test_good, test_bad),
        "heldout_baseline_always_" + favourite: sum(r["chosen_style"] == favourite for r in test_rows) / len(test_rows),
        "your_choices": dict(Counter(r["chosen_style"] for r in records)),
        "final_loss": round(loss.item(), 4), "encoder": ENCODER_ID, "answer_model": OLLAMA_MODEL,
    }
    print(json.dumps(report, indent=2))
    print("\nheld-out questions: what you chose vs what the head prefers")
    with torch.no_grad():
        for r, g, b in zip(test_rows, test_good, test_bad):
            head_choice = r["chosen_style"] if head(g).item() > head(b).item() else \
                          ("detailed" if r["chosen_style"] == "short" else "short")
            print(f"  you: {r['chosen_style']:<8}  head: {head_choice:<8}  {r['question'][:70]}")
    torch.save({"state_dict": head.state_dict(), "encoder": ENCODER_ID, "evidence_tokens": EVIDENCE_TOKENS,
                "reward_mean": reward_mean, "reward_std": reward_std}, DATA / "reward-head.pt")
    (DATA / "reward-report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"\nsaved to {DATA}: reward-head.pt, reward-report.json")

{
  "train_pairs": 9,
  "heldout_pairs": 3,
  "train_preference_accuracy": 1.0,
  "heldout_before_training": 0.3333333432674408,
  "heldout_after_training": 1.0,
  "heldout_baseline_always_short": 1.0,
  "your_choices": {
    "short": 10,
    "detailed": 2
  },
  "final_loss": 0.1351,
  "encoder": "Qwen/Qwen2.5-0.5B-Instruct",
  "answer_model": "gemma4:12b"
}

held-out questions: what you chose vs what the head prefers
  you: short     head: short     What does CTK303 cover?
  you: short     head: short     Tell me about ENG 101.
  you: short     head: short     What is IT-344 about?

saved to C:\Users\Jake\Downloads\ISU-RAG-VSCode\isu_catalog_cache\experiments\rlhf: reward-head.pt, reward-report.json


---
## 6 · Use the reward: score fresh answers and show the preferred one

For a new question, Gemma 4 writes both styles from ReggieBot's sources, the trained head scores each, and the
higher score wins. This is **best-of-2**: the reward decides what a user sees without changing Gemma 4. A score
is a relative preference signal, not a probability and not a guarantee that the answer is correct. Compare the
choice with your own judgment, and look closely where they disagree.

In [6]:
TRY_QUESTION = "How do I set up direct deposit for my refund?"


@torch.no_grad()
def learned_reward(item, answer):
    score = head(reward_features(item["question"], item["evidence"], answer)).item()
    return (score - reward_mean) / reward_std                  # scaled by the training pairs only


def rlhf_answer(question):
    """Both styles from the same sources; return the one the learned reward prefers."""
    item = retrieve(question)
    candidates = {style: write(item, style) for style in STYLES}
    scores = {style: learned_reward(item, text) for style, text in candidates.items()}
    for style, text in candidates.items():
        print(f"{STYLES[style][0]:<22} learned score {scores[style]:+.2f}   ({len(text.split())} words)")
    best = max(scores, key=scores.get)
    print(f"\nReggieBot shows {STYLES[best][0]}:\n\n{candidates[best]}\n\nSources:\n{sources_markdown(item)}")
    return best, candidates[best]


if head is None:
    print("Train the reward head in §5 first.")
else:
    rlhf_answer(TRY_QUESTION)

A · short and simple   learned score -0.97   (86 words)
B · explains more      learned score -0.76   (191 words)

ReggieBot shows B · explains more:

To set up direct deposit (eRefunds) for your refund, you must complete the process through your student login. Please note that only one refund account may exist at a time, and eRefunds are not available for Parent Plus Loans that are not released to the student.

Follow these steps to complete the setup:
*   Log in to My Illinois State and select the **Finance** tab from the top navigation bar.
*   Scroll down to the **Student Accounts** section and click "**eRefunds (Direct Deposit)**."
*   Select "**Electronic Refunds**" from the menu on the right.
*   Enroll in **Two-Step Verification** to add the eRefund. Click the **Security Settings** link and follow the on-screen steps.
*   Once the passcode is verified, click on the **Payment Profile** tab to complete the setup.
*   You may select an existing saved payment method or choose "**Set

---
## Discuss

* Does training accuracy imply held-out agreement? Did the head beat the "always your favourite style" baseline?
* Does a high learned score mean the answer is correct? What would a confident wrong answer score?
* Why does the reward input include the evidence, and not just the answer?
* A is always on the left and short. Could where an answer sits, or its length, sway your choices?
* Which weights would have to change for Gemma 4 itself to write in the style you prefer, without best-of-2?

References: [pairwise reward-model objective (TRL RewardTrainer)](https://huggingface.co/docs/trl/reward_trainer),
[Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).